# Battery cross-cell transfer, v2 — best structure, and a real search

## What v1 found, read against the right baseline

v1 scored transfer against the **flat line**, 0.02927. That was the wrong bar, and it flattered every
condition. On this cell you can extrapolate the fade curve from the 20 cycles before the origin using
no network at all, and score **0.01439**. That is the number any model has to beat to justify existing.

| forward: train B0006 -> forecast B0005 | RMSE | |
|---|---|---|
| flat line | 0.02927 | the bar v1 used |
| **fade curve alone, no network** | **0.01439** | the bar it should have used |
| A1 zero-shot network | 0.02974 | no better than persistence |
| A2 fade + B0006 residual net | 0.01998 | **39% worse than the curve by itself** |
| A3 zero-shot + delta calibration | 0.03367 | worse than A1 |
| B1 within-cell plain | 0.01135 | reproduces P1 (0.01095) |
| physics within-cell | 0.00850 | our actual best — absent from v1's ladder |

Reverse is worse: the B0005 residual net on B0006 gives 0.04390 against a curve-alone value of
0.01246. A1 "beating persistence" there (0.03487 vs 0.04608) is an artefact of B0006 fading fast
enough to make a flat line a very weak baseline.

**So nothing transferred.** In both directions the imported network subtracts value from a curve fit
that costs nothing. Two of my framing assumptions were also wrong and should not survive into the
write-up: B0006 does **not** fade faster near the origin (-0.001697/cycle against B0005's
-0.001833), and A3's rescaling therefore amplified deltas by 1.081 rather than damping them.

## Why this notebook exists

Two things were missing from v1, and both are mine.

1. Every condition ran on **round-1 default hyperparameters**. No BO, no PSO, nothing optimised.
2. The within-cell control was the *plain* model, 0.01135, not the physics model at 0.00850. The cost
   of transfer was therefore measured against the wrong incumbent, and is larger than v1 implied.

The honest reason optimisation has never been run on this dataset is that it never had a working
selection signal. The protocol study measured it directly: across 18 deterministic fade
configurations, **Spearman(validation, test) = -0.078, p = 0.757** — choosing on validation was no
better than choosing at random. B0005 has 168 cycles, 123 before the origin and 35 after. There is
no room to carve a validation set that predicts the forecast region.

**A second cell is what fixes that**, and it is the real reason to keep going after a negative
transfer result. B0006 can be used as a complete, independent selection instrument: train *and*
validate there, then apply to B0005 without ever touching it. That is a cleaner model-selection
protocol than anything available within a single cell, and we have never had it before.

So this notebook searches, and the headline output is not an RMSE. It is
**Spearman(source-cell validation, target-cell test)** — the same diagnostic that settled N-CMAPSS,
where +0.822 on a good representation and +0.276 on a bad one explained years of confusing results.
If it is near zero here, the battery genuinely cannot be optimised by search and we can finally say
so with a measurement instead of a shrug.

## Structure

Best structure, per the standing rule: the **physics-informed model** — fade curve anchored on the
target cell's own pre-origin cycles, network predicting the residual. That is the 0.00850
configuration, not the plain one.

Three arms at equal budget (BO, PSO, random control), 10 hyperparameters including window length.
Two candidate selection signals measured for every trial:

- `val_src` — recursive rollout on **B0006's own** forecast region, source cell only
- `val_tgt` — recursive rollout on the last stretch of **B0005's pre-origin** cycles

and `test` — the real B0005 forecast region, recorded for every trial and **never** used to select
anything. Which of the two signals actually predicts test is the question; picking the winner by test
would answer nothing.

In [ ]:
import os, gc, time, json, math, random, warnings
os.environ['PYTHONHASHSEED']='42'
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from scipy.io import loadmat
from scipy.optimize import curve_fit
from scipy.stats import spearmanr, ttest_1samp, norm
from scipy.stats.qmc import Sobol
from sklearn.metrics import mean_squared_error
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g,True)
    except RuntimeError as _e: print('memory growth:',_e)
from tensorflow.keras import backend as K
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Bidirectional, Concatenate, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
def preflight():
    if not tf.config.list_physical_devices('GPU'): print('CPU mode'); return False
    try:
        x=np.random.rand(8,10,1).astype('float32'); y=np.random.rand(8,1).astype('float32')
        i=Input(shape=(10,1)); o=Dense(1)(Bidirectional(LSTM(4))(i)); m=Model(i,o)
        m.compile('adam','mse'); m.fit(x,y,epochs=1,verbose=0); K.clear_session()
        print('GPU preflight passed'); return True
    except Exception as e:
        print('GPU preflight FAILED (%s) -> CPU'%type(e).__name__); K.clear_session(); return False
USE_GPU=preflight(); DEVICE='/GPU:0' if USE_GPU else '/CPU:0'
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('determinism:',e)
SEED=42; EPOCHS,PATIENCE,BATCH=300,30,16
FADE_L=20; N_ARM=30; N_INIT=10; SEEDS_SEARCH=2; SEEDS_FINAL=5; SHORTLIST=4
SRC_NAME,TGT_NAME='B0006','B0005'
print('TF',tf.__version__,'| device',DEVICE)

In [ ]:
def load_cell(name):
    mat=loadmat(name+'.mat'); cyc=mat[name][0,0]['cycle'][0]; caps=[]
    for i in range(len(cyc)):
        r=cyc[i]
        if r['type'][0]!='discharge': continue
        caps.append(float(r['data'][0][0]['Capacity'][0][0]))
    CAP=np.asarray(caps,np.float64); S=(CAP/CAP[0]).astype(np.float32)
    N=len(S); o=int(0.8*N)-1
    return dict(name=name,S=S,N=N,o=o,H=N-o)
f_lin=lambda k,a,b: a*k+b
def fade_curve(S,N,origin,L=FADE_L):
    a=max(0,origin-L); par,_=curve_fit(f_lin,np.arange(a,origin,dtype=float),S[a:origin],
                                       p0=[-1e-3,1.0],maxfev=20000)
    return np.clip(f_lin(np.arange(float(N)),*par),0,1.2).astype(np.float32), par

SRC,TGT=load_cell(SRC_NAME),load_cell(TGT_NAME)
for c in (SRC,TGT):
    c['CURVE'],c['par']=fade_curve(c['S'],c['N'],c['o'])
    c['RESID']=(c['S']-c['CURVE']).astype(np.float32)
    c['floor']=float(np.sqrt(mean_squared_error(c['S'][c['o']:],np.full(c['H'],c['S'][c['o']-1]))))
    c['curve_only']=float(np.sqrt(mean_squared_error(c['S'][c['o']:],c['CURVE'][c['o']:])))
# an inner origin on the TARGET's pre-origin region -> the val_tgt signal, no test data touched
TGT['o_in']=TGT['o']-20; TGT['H_in']=20
TGT['CURVE_IN'],_=fade_curve(TGT['S'],TGT['N'],TGT['o_in'])
TGT['RESID_IN']=(TGT['S']-TGT['CURVE_IN']).astype(np.float32)
SRC['o_in'],SRC['H_in']=SRC['o'],SRC['H']          # val_src uses the source cell's real forecast region

print('%-7s %4s %7s %9s %11s %11s'%('cell','N','origin','horizon','flat line','curve alone'))
for c in (SRC,TGT): print('%-7s %4d %7d %9d %11.5f %11.5f'%(c['name'],c['N'],c['o'],c['H'],c['floor'],c['curve_only']))
print('\nTHE BAR: %.5f  (fade curve on %s, no network at all)'%(TGT['curve_only'],TGT_NAME))
print('reference points: within-cell physics 0.00850 | within-cell plain 0.01135 | v1 best transfer 0.01998')
PHYS_INCUMBENT=0.00850

## Search space and the physics-transfer evaluator

In [ ]:
KSETS=[(3,),(3,5),(3,5,7),(5,7,11)]
SPACE=[('window','int',5,20), ('filters','int',8,96), ('kset','cat',[0,1,2,3]),
       ('n_cnn','int',1,2), ('lstm_units','int',8,96), ('bidir','cat',[0,1]),
       ('n_dense','int',1,2), ('dense_units','int',16,160),
       ('dropout','float',0.0,0.4), ('lr','log',1e-4,1e-2)]
D=len(SPACE)
def decode(z):
    z=np.clip(np.asarray(z,float),0,1); out={}
    for zi,s in zip(z,SPACE):
        n,k=s[0],s[1]
        if k=='int': out[n]=int(round(s[2]+zi*(s[3]-s[2])))
        elif k=='float': out[n]=float(s[2]+zi*(s[3]-s[2]))
        elif k=='log': out[n]=float(np.exp(np.log(s[2])+zi*(np.log(s[3])-np.log(s[2]))))
        else: out[n]=s[2][min(int(zi*len(s[2])),len(s[2])-1)]
    return out
def encode(kw):
    z=[]
    for s in SPACE:
        n,k=s[0],s[1]; v=kw[n]
        if k in ('int','float'): z.append((v-s[2])/(s[3]-s[2]))
        elif k=='log': z.append((np.log(v)-np.log(s[2]))/(np.log(s[3])-np.log(s[2])))
        else: z.append((s[2].index(v)+0.5)/len(s[2]))
    return np.clip(np.array(z,float),0,1)
def nrmsd(a,b): return float(np.linalg.norm(np.asarray(a)-np.asarray(b))/np.sqrt(D))
BASE=dict(window=10,filters=32,kset=2,n_cnn=1,lstm_units=32,bidir=1,n_dense=1,
          dense_units=64,dropout=0.1,lr=1e-3)   # round-1 defaults = the 0.00850 structure
Z_BASE=encode(BASE)
print('dimension',D,'| round-trip', all(decode(Z_BASE)[k]==BASE[k] or abs(decode(Z_BASE)[k]-BASE[k])<1e-9
                                        for k in BASE))

def build(kw,seed=SEED):
    ki=dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed))
    inp=Input(shape=(kw['window'],1)); c=inp
    for _ in range(kw['n_cnn']):
        c=Concatenate()([Conv1D(kw['filters'],k,padding='same',activation='relu',**ki)(c)
                         for k in KSETS[kw['kset']]]) if len(KSETS[kw['kset']])>1 else \
          Conv1D(kw['filters'],KSETS[kw['kset']][0],padding='same',activation='relu',**ki)(c)
        c=Dropout(kw['dropout'],seed=seed)(c)
        if int(c.shape[1])>2: c=MaxPooling1D(2,padding='same')(c)
    c=GlobalAveragePooling1D()(c)
    r=Bidirectional(LSTM(kw['lstm_units'],**ki))(inp) if kw['bidir'] else LSTM(kw['lstm_units'],**ki)(inp)
    h=Concatenate()([c,r])
    for _ in range(kw['n_dense']):
        h=Dense(kw['dense_units'],activation='relu',**ki)(h); h=Dropout(kw['dropout'],seed=seed)(h)
    m=Model(inp,Dense(1,activation='linear',**ki)(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(kw['lr']),loss='mse'); return m

def windows(series,idxs,W):
    X=np.asarray([series[e-W:e] for e in idxs],np.float32).reshape(-1,W,1)
    y=np.asarray([series[e]-series[e-1] for e in idxs],np.float32).reshape(-1,1)
    return X,y
def rollout(m,resid,origin,H,curve,W):
    win=list(resid[origin-W:origin]); out=[]; prev=float(resid[origin-1])
    for _ in range(H):
        x=np.asarray(win[-W:],np.float32).reshape(1,W,1)
        prev=float(prev+float(m(x,training=False).numpy().ravel()[0]))
        out.append(prev); win.append(prev)
    return np.clip(curve[origin:origin+H]+np.asarray(out,np.float32),0.0,1.2)

def train_resid(kw,seed,series,idxs):
    X,y=windows(series,idxs,kw['window']); c=int(0.85*len(X))
    np.random.seed(seed); tf.random.set_seed(seed)
    with tf.device(DEVICE):
        m=build(kw,seed)
        m.fit(X[:c],y[:c],validation_data=(X[c:],y[c:]),epochs=EPOCHS,batch_size=BATCH,
              verbose=0,shuffle=True,
              callbacks=[EarlyStopping('val_loss',patience=PATIENCE,restore_best_weights=True),
                         ReduceLROnPlateau('val_loss',factor=0.5,patience=10,min_lr=1e-7)])
    return m
rmse=lambda a,b: float(np.sqrt(mean_squared_error(a,b)))

TRIALS=[]
def evaluate(kw,tag='',seeds=SEEDS_SEARCH):
    """val_src and val_tgt are candidate SELECTION signals. test is recorded, never returned."""
    W=kw['window']; vs,vt,te=[],[],[]
    for s in range(seeds):
        sd=SEED+s
        try:
            idx_src=[e for e in range(W,SRC['o'])]
            m1=train_resid(kw,sd,SRC['RESID'],idx_src)
            vs.append(rmse(SRC['S'][SRC['o']:],rollout(m1,SRC['RESID'],SRC['o'],SRC['H'],SRC['CURVE'],W)))
            idx_all=[e for e in range(W,SRC['N'])]
            m2=train_resid(kw,sd,SRC['RESID'],idx_all)
            vt.append(rmse(TGT['S'][TGT['o_in']:TGT['o_in']+TGT['H_in']],
                           rollout(m2,TGT['RESID_IN'],TGT['o_in'],TGT['H_in'],TGT['CURVE_IN'],W)))
            te.append(rmse(TGT['S'][TGT['o']:],rollout(m2,TGT['RESID'],TGT['o'],TGT['H'],TGT['CURVE'],W)))
        except Exception as e:
            print('   trial failed (%s)'%type(e).__name__); vs.append(9.9); vt.append(9.9); te.append(np.nan)
        K.clear_session(); gc.collect()
    r=dict(tag=tag,val_src=float(np.mean(vs)),val_tgt=float(np.mean(vt)),
           test=float(np.nanmean(te)),test_sd=float(np.nanstd(te)),**kw)
    TRIALS.append(r); return r['val_src']

t0=time.time(); v=evaluate(BASE,'baseline'); b=TRIALS[-1]
print('baseline (round-1 structure): val_src %.5f | val_tgt %.5f | test %.5f | %.0fs'
      %(b['val_src'],b['val_tgt'],b['test'],time.time()-t0))
print('estimated: %.0f min for %d search trials'%(3*N_ARM*(time.time()-t0)/60,3*N_ARM))

## Three arms — random control, BO, PSO

In [ ]:
def far_design(n,refs,min_d=0.35,seed=SEED):
    eng=Sobol(d=D,scramble=True,seed=seed); pts=[]; g=0
    while len(pts)<n and g<50:
        g+=1
        for c in eng.random(max(8,n*4)):
            if min(nrmsd(c,r) for r in refs)>=min_d:
                pts.append(c)
                if len(pts)==n: break
    rs=np.random.RandomState(seed)
    while len(pts)<n:
        c=rs.rand(D)
        if min(nrmsd(c,r) for r in refs)>=min_d: pts.append(c)
    return np.array(pts[:n])

print('--- RANDOM (control) ---'); t0=time.time()
for i,z in enumerate(far_design(N_ARM,[Z_BASE],seed=SEED+1)):
    y=evaluate(decode(z),'random')
    if (i+1)%10==0: print('  %2d/%d best val_src %.5f'%(i+1,N_ARM,min(t['val_src'] for t in TRIALS if t['tag']=='random')))
print('random done %.1f min'%((time.time()-t0)/60))

print('--- BO ---'); t0=time.time()
Zobs=list(far_design(N_INIT,[Z_BASE],seed=SEED+2)); Yobs=[evaluate(decode(z),'bo_init') for z in Zobs]
Zobs.append(Z_BASE.copy()); Yobs.append(b['val_src'])
def ei(mu,sd,best,xi=0.01):
    sd=np.maximum(sd,1e-9); imp=best-mu-xi; z=imp/sd
    return imp*norm.cdf(z)+sd*norm.pdf(z)
for it in range(N_ARM-N_INIT):
    Za,Ya=np.array(Zobs),np.array(Yobs); ok=Ya<9
    Zf,Yf=(Za[ok],Ya[ok]) if ok.sum()>=4 else (Za,Ya)
    gp=GaussianProcessRegressor(ConstantKernel(1.0)*Matern(length_scale=np.ones(D),nu=2.5)+WhiteKernel(1e-4),
                                normalize_y=True,n_restarts_optimizer=3,random_state=SEED).fit(Zf,Yf)
    cand=np.random.RandomState(SEED+it).rand(3000,D); mu,sd=gp.predict(cand,return_std=True)
    zb=cand[int(np.argmax(ei(mu,sd,Yf.min())))]
    Yobs.append(evaluate(decode(zb),'bo')); Zobs.append(zb)
print('BO done %.1f min | best val_src %.5f'%((time.time()-t0)/60,min(Yobs)))

print('--- PSO ---'); t0=time.time()
SW=6; IT=N_ARM//SW; rng=np.random.RandomState(SEED+7)
P=far_design(SW,[Z_BASE],seed=SEED+7); P[0]=Z_BASE.copy()
V=(rng.rand(SW,D)-0.5)*0.3; pb=P.copy(); pby=np.full(SW,np.inf); gb=None; gby=np.inf
for it in range(IT):
    w=0.85-(0.5)*it/max(1,IT-1)
    for k in range(SW):
        y=b['val_src'] if (it==0 and k==0) else evaluate(decode(P[k]),'pso')
        if it==0 and k==0: TRIALS.append(dict(TRIALS[0],tag='pso'))
        if y<pby[k]: pby[k]=y; pb[k]=P[k].copy()
        if y<gby: gby=y; gb=P[k].copy()
    V=np.clip(w*V+1.5*rng.rand(SW,D)*(pb-P)+1.5*rng.rand(SW,D)*(gb-P),-0.25,0.25); P=np.clip(P+V,0,1)
    print('  iter %d/%d gbest %.5f'%(it+1,IT,gby))
print('PSO done %.1f min'%((time.time()-t0)/60))
T=pd.DataFrame(TRIALS); T.to_csv('battery_transfer_v2_trials.csv',index=False)

## The question this notebook was built to answer

In [ ]:
S_=T[T.tag.isin(['random','bo','bo_init','pso'])].dropna(subset=['test'])
S_=S_[(S_.val_src<9)&(S_.val_tgt<9)]
r1,p1=spearmanr(S_.val_src,S_.test); r2,p2=spearmanr(S_.val_tgt,S_.test)
print('='*78)
print('CAN THE BATTERY BE OPTIMISED BY SEARCH?   n = %d configurations'%len(S_))
print('='*78)
print('  Spearman(val_src  [B0006 forecast region], test) = %+.3f   p = %.4g'%(r1,p1))
print('  Spearman(val_tgt  [B0005 pre-origin]     , test) = %+.3f   p = %.4g'%(r2,p2))
print()
print('  for reference: N-CMAPSS matched protocol +0.822 | N-CMAPSS old representation +0.276')
print('                 battery fade-config study    -0.078 (p=0.757, selection no better than random)')
best_sig='val_src' if abs(r1)>abs(r2) else 'val_tgt'
usable = (r1>0.3 and p1<0.05) or (r2>0.3 and p2<0.05)
print('\n  verdict: %s'%('a usable selection signal exists -> search is meaningful here'
      if usable else 'NO usable selection signal -> search cannot rank configurations on this dataset'))
print('  stronger of the two: %s'%best_sig)

print('\n--- arm comparison at equal budget (ranked by %s) ---'%best_sig)
S_['arm']=S_.tag.replace({'bo_init':'BO','bo':'BO'})
for a in ['random','BO','pso']:
    d=S_[S_.arm==a]
    if len(d):
        w=d.nsmallest(1,best_sig).iloc[0]
        print('  %-7s n=%2d  best %s %.5f  -> its test %.5f'%(a,len(d),best_sig,w[best_sig],w['test']))

In [ ]:
short=S_.nsmallest(SHORTLIST,best_sig)
print('shortlist by %s (test column shown but NOT used to choose):'%best_sig)
print(short[['arm',best_sig,'val_src','val_tgt','test','window','filters','lstm_units','dense_units']].to_string(index=False))
def kw_of(row):
    g=(lambda n: getattr(row,n)) if hasattr(row,'_fields') else (lambda n: row[n])
    return {s[0]:(int(g(s[0])) if s[1] in ('int','cat') else float(g(s[0]))) for s in SPACE}
FIN=[]
for nm,kw in [('round-1 baseline',BASE)]+[('search%d'%(i+1),kw_of(r)) for i,r in enumerate(short.itertuples())]:
    evaluate(kw,'final:'+nm,seeds=SEEDS_FINAL); f=TRIALS[-1]
    FIN.append(dict(name=nm,**{k:f[k] for k in ('val_src','val_tgt','test','test_sd')},**kw))
    print('  %-18s test %.5f +/- %.5f'%(nm,f['test'],f['test_sd']))
F=pd.DataFrame(FIN); F.to_csv('battery_transfer_v2_final.csv',index=False)

pick=F[F.name!='round-1 baseline'].nsmallest(1,best_sig).iloc[0]
bar=TGT['curve_only']
print('\n'+'='*78)
print('%-46s %10s %8s'%('','RMSE','vs bar'))
for lab,v in [('flat line (v1 used this bar)',TGT['floor']),
              ('FADE CURVE ALONE, no network  <-- THE BAR',bar),
              ('v1 transfer, round-1 structure',0.01998),
              ('v2 transfer, searched structure',pick['test']),
              ('within-cell plain (P1)',0.01135),
              ('within-cell physics (incumbent)',PHYS_INCUMBENT)]:
    print('%-46s %10.5f %+8.1f%%'%(lab,v,100*(v-bar)/bar))
print('='*78)
verdict=('transfer now ADDS value over the curve' if pick['test']<bar
         else 'transfer still SUBTRACTS value: the curve alone is better')
print('verdict:',verdict)
json.dump(dict(spearman_val_src=float(r1),spearman_val_tgt=float(r2),usable=bool(usable),
               bar_curve_only=bar,flat_line=TGT['floor'],v1_transfer=0.01998,
               v2_transfer=float(pick['test']),within_cell_physics=PHYS_INCUMBENT,verdict=verdict),
          open('battery_transfer_v2_summary.json','w'),indent=2)
print('\nsaved battery_transfer_v2_trials.csv, _final.csv, _summary.json in',os.getcwd())